# `tab:estimator-gap`, mean performance under the two log1p estimators (A vs B)

Not a figure, a TABLE, but it belongs with the paper outputs because it is read next to `fig:mean-performance` and shares its cohort, its gene ordering and its cutoffs.

**What A and B are.** A count head is fitted on integer counts at the common depth $T$, so putting it back on the log1p-CP10k axis is not unique. With $\delta = T/10^4$:

| | estimator | uses | code / artifact name |
|---|---|---|---|
| $A$ | $\log(1 + \mu/\delta)$, plug-in / mean-only | the fitted mean $\mu$ ONLY, ignores $\alpha$ | `{metric}_logmean`, e.g. `pearson_logmean` |
| $B$ | $\mathbb{E}[\log(1+Y/\delta)] = \sum_k \log(1+k/\delta)P(Y=k)$, predictive mean of the transform | $\mu$ AND the dispersion $\alpha$ | `g*`, the headline `pearson` column, `gstar_fold{k}.h5ad` |

They differ by the Jensen gap of the concave $\log(1+\cdot)$, which is NOT a constant offset: it grows with the predictive spread. $B$ is the exact analogue of the estimand of a Gaussian trained on that scale; $A$ is also the scale the count models early-stop on.

**The comparison is WITHIN one count model.** A Gaussian predicts on the log1p axis directly and has a single estimator, so it has no $A$ and cannot enter this table; the mean-only baseline has no $A$ either, which is why the gap is never expressed against it.

**No recomputation.** Both estimators already sit side by side in every count run: `{metric}` (B) and `{metric}_logmean` (A) are two columns of the same `atoms.parquet`, carried into canal1's `benchmark_summary.parquet`. `estimator_gap` (`src/evaluation/benchmark_summary.py`) builds $D = B - A$ per (gene, donor) and collapses it the same two ways as `paired_deltas_full`: over genes to K donor points (Student-$t$), and over donors to N gene points (percentile bootstrap).

**Cohort and axis.** `pr_classic3`, the same run as `mean_performance_classic.ipynb`, and the SAME shared gene ordering: `CP_MODELS` (all eight heads) is loaded so the ranking is identical to the figures, while the table itself shows only the count heads that have an $A$. log1p-CP10k axis only; the depth-blind `*_raw` pair reports on log1p(raw counts) and must never share a table with these.

In [1]:
import sys
import importlib
from pathlib import Path
import numpy as np
import pandas as pd

PROJ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.append(str(PROJ))   # parent dir of src (notebooks are not pip-installed on the cluster)

import src.evaluation.benchmark_summary as bs
importlib.reload(bs)
from src.evaluation.benchmark_summary import discover_run_summaries, estimator_gap

RUN_ID    = "pr_classic3"
DATASETS  = ["10x_TuPro", "TLS", "LUNG_CANCER"]
RADIUS    = "r75"
NS        = [50, 100, 500, 1000, 2000, None]   # None -> the full shared panel

# Loaded, NOT all shown. The eight heads are what fixes the shared gene ordering in the paper
# figures, and estimator_gap rebuilds that ordering from whatever is passed to it -- so loading
# the same eight is what makes a given n denote the same gene set here as on fig:mean-performance.
CP_MODELS = ["msemeanonly", "homoscedastic", "heteroscedastic", "spot_homoscedastic",
             "poisson_cp", "negativebinomial_hetero_noanchor_cp",
             "negativebinomial_spotdisp_cp", "negativebinomial_genedisp_cp"]

# The count heads the TABLE shows, printed AND LaTeX. The two restricted-dispersion NB heads
# (spotdisp, genedisp) are out of every paper output, exactly as PLOT_MODELS excludes them in
# mean_performance_classic.ipynb -- they only carry the parametrization argument.
# They stay in CP_MODELS above for the gene ORDERING only, never shown, so a given n keeps
# denoting the same gene set here as on the paper figures.
TABLE_MODELS = ["poisson_cp", "negativebinomial_hetero_noanchor_cp"]

MODEL_LABELS = {
    "poisson_cp":                          "Poisson",
    "negativebinomial_hetero_noanchor_cp": "Negative binomial",
}
DATASET_LABELS = {"10x_TuPro":   "Tumor Profiler",
                  "TLS":         "Kidney/Lung with TLS",
                  "LUNG_CANCER": "NSCLC"}

TEXDIR = PROJ / "results" / "_analysis" / "figures_paper"
TEXDIR.mkdir(parents=True, exist_ok=True)

summaries = discover_run_summaries(PROJ / "results", RUN_ID, DATASETS,
                                   radius=RADIUS, models=CP_MODELS, lambda_keep=0.0)
print(f"{len(summaries)} runs charges")
for s in summaries:
    print(f"   {s['dataset']:<12s} {s['label']}")

# Without the count heads there is no A at all and every table below is silently empty; fail here
# instead -- most likely cause is that canal1 has not yet written benchmark_summary.parquet.
_found = {s["label"].split("|")[0] for s in summaries}
_missing = [m for m in TABLE_MODELS if m not in _found]
assert not _missing, (
    f"count heads absent from the loaded runs: {_missing} (present: {sorted(_found)}). "
    f"did canal1 run on them?")

24 runs charges
   10x_TuPro    heteroscedastic|lam0.0
   10x_TuPro    homoscedastic|lam0.0
   10x_TuPro    msemeanonly
   10x_TuPro    negativebinomial_genedisp_cp
   10x_TuPro    negativebinomial_hetero_noanchor_cp
   10x_TuPro    negativebinomial_spotdisp_cp
   10x_TuPro    poisson_cp
   10x_TuPro    spot_homoscedastic|lam0.0
   TLS          heteroscedastic|lam0.0
   TLS          homoscedastic|lam0.0
   TLS          msemeanonly
   TLS          negativebinomial_genedisp_cp
   TLS          negativebinomial_hetero_noanchor_cp
   TLS          negativebinomial_spotdisp_cp
   TLS          poisson_cp
   TLS          spot_homoscedastic|lam0.0
   LUNG_CANCER  heteroscedastic|lam0.0
   LUNG_CANCER  homoscedastic|lam0.0
   LUNG_CANCER  msemeanonly
   LUNG_CANCER  negativebinomial_genedisp_cp
   LUNG_CANCER  negativebinomial_hetero_noanchor_cp
   LUNG_CANCER  negativebinomial_spotdisp_cp
   LUNG_CANCER  poisson_cp
   LUNG_CANCER  spot_homoscedastic|lam0.0


## 1. The table, printed

One block per dataset, columns = the cutoffs of `NS` (the last one is the full shared panel). Three readings per count head:

- **$B$** and **$A$**: the mean performance itself, i.e. the cumulative slide-aware Pearson on log1p-CP10k over the first $n$ genes of the shared ordering, averaged over the held-out donors. These are the two numbers the paper compares.
- **gap $B-A$**: the paired difference, built per (gene, donor) BEFORE any averaging, so it is not the difference of two independently-averaged columns. Positive = routing the prediction through the full fitted predictive beats using the mean alone.
- **half-widths**: donor Student-$t$ CI / donor SEM / gene bootstrap CI, the same three the other paper outputs report. `*` marks a donor $t$-CI excluding zero, `o` a gene bootstrap CI excluding zero. The two answer different questions (a new patient vs a new gene) and are shown side by side rather than one being picked.

Poisson is the control: its predictive has no free dispersion, so its gap is expected to be near zero, and a non-trivial NB gap is what says the dispersion is doing work on this axis.

In [2]:
gaps = estimator_gap(summaries, metric_b="pearson", metric_a="pearson_logmean",
                     ns=NS, rank_metric="pearson")
# The two restricted-dispersion NB heads only served to RANK the genes (via CP_MODELS,
# cell 2): cut here, so neither printed nor exported, as in the other paper outputs.
gaps = gaps[gaps["model"].isin(TABLE_MODELS)]
print(f"{len(gaps)} rows (dataset x count head x cutoff)\n")

for ds in DATASETS:
    sub = gaps[gaps["dataset"] == ds]
    if sub.empty:
        continue
    print(f"=== {DATASET_LABELS.get(ds, ds)} ({ds}) " + "=" * 30)

    print("\n  B  predictive mean of the transform (g*, uses mu AND alpha)")
    print(sub.pivot_table(index="model", columns="n", values="B_mean",
                          aggfunc="first").round(4).to_string())
    print("\n  A  plug-in mean-only, log1p(mu/delta) (pearson_logmean, ignores alpha)")
    print(sub.pivot_table(index="model", columns="n", values="A_mean",
                          aggfunc="first").round(4).to_string())

    cells = [f"{r.gap_mean:+.4f}{'*' if r.excludes_zero_t else ' '}"
             f"{'o' if r.excludes_zero_gene else ' '}" for r in sub.itertuples()]
    print("\n  gap B - A  ('*' = donor t-CI excludes 0, 'o' = gene bootstrap excludes 0)")
    print(sub.assign(c=cells).pivot_table(index="model", columns="n",
                                          values="c", aggfunc="first").to_string())

    w = [f"{(r.gap_ci_hi - r.gap_ci_lo) / 2:.4f}/{r.gap_sem:.4f}/"
         f"{(r.gene_ci_hi - r.gene_ci_lo) / 2:.4f}" for r in sub.itertuples()]
    print("\n  half-widths  donor t-CI / donor SEM / gene bootstrap")
    print(sub.assign(c=w).pivot_table(index="model", columns="n",
                                      values="c", aggfunc="first").to_string())
    print(f"\n  K donors = {int(sub['K'].iloc[0])}, "
          f"shared panel = {int(sub['n'].max())} genes\n")

36 rows (dataset x count head x cutoff)

=== Tumor Profiler (10x_TuPro) ==============================

  B  predictive mean of the transform (g*, uses mu AND alpha)
n                                      50      100     500     1000    2000    3000
model                                                                              
negativebinomial_hetero_noanchor_cp  0.5412  0.5002  0.3766  0.3132  0.2308  0.1655
poisson_cp                           0.5386  0.5016  0.3863  0.3223  0.2337  0.1653

  A  plug-in mean-only, log1p(mu/delta) (pearson_logmean, ignores alpha)
n                                      50      100     500     1000    2000    3000
model                                                                              
negativebinomial_hetero_noanchor_cp  0.5360  0.4985  0.3812  0.3171  0.2284  0.1609
poisson_cp                           0.5384  0.5014  0.3861  0.3222  0.2337  0.1654

  gap B - A  ('*' = donor t-CI excludes 0, 'o' = gene bootstrap excludes 0)
n          

## 2. The table

What the paper reports:*mean accuracy under estimator $A$ against estimator $B$, per cohort and per count head, on the log1p-CP10k scale, with the per-donor interval*.

Shape: **cutoffs as rows, three columns $B$, $A$, $B-A$.**

| Dataset | Model | $n$ | $B$ | $A$ | $B-A$ |
|---|---|---|---|---|---|
| Tumor | Poisson | 50 | . | . | . |
| Profiler | | 500 | . | . | . |
| | | all | . | . | . |
| | Negative binomial | 50 | . | . | . |
| ... | | | | | |

18 rows, 3 numeric columns, every cell carrying `value` $\pm$ `half-width`, everything at 3 decimals.

**The two intervals are different, one per column type, and that is the point of the third column.**

| column | interval | why it is that one |
|---|---|---|
| $B$, $A$ | **level**, between donors | how that accuracy would move on another patient. It is a cohort property and it is wide |
| $B-A$ | **paired**, within donor | each donor contributes $B-A$ over its own genes, so the between-donor variation cancels. One to two orders tighter |

Printing the paired interval in its own column is what stops a reader concluding, from the two overlapping intervals on its left, that the difference is not real. Those measure different things.

**Bold = one condition, on the value as displayed: $|\mathrm{round}(B-A,\,3)| \ge$ `BOLD_MIN_GAP` (0.005).** Rounded, not raw, deliberately. On the raw value TuPro NB bolded `+0.005` at $n=50$ (truly $+0.0052$) but not `-0.005` at $n=500$ (truly $-0.0046$): two cells printing the same number, treated differently on a 4th decimal the reader cannot see. Deciding on the displayed value makes every bold checkable against the table itself. It remains a magnitude cut and not a significance test; the $B-A$ column next to it carries the evidence, and the twin below the table prints the raw gap, the displayed gap and whether each is resolved against its paired interval.

**Dataset labels spread over the rows the block already has.** `DATASET_TEX_LINES` gives the lines, and `to_latex` puts the first on the `n=50` row and the second on the `n=500` row. Two earlier attempts were worse: a `p{}` column reserves its width whatever the label, leaving white space beside the short ones, and both `p{}` and a nested `tabular` make the `n=50` row twice as tall, which opens a visible gap between `n=50` and `n=500`. Using the existing rows costs no height at all and keeps the column at the width of `Kidney/Lung`. A three-row block therefore accepts up to three label lines. LaTeX only: the markdown render of section 3 keeps the single-line `DATASET_LABELS`.

A gap that rounds to zero prints as `0.000`, unsigned, since `-0.000` means nothing. `NS3` controls the cutoff rows; `[None]` alone gives the full panel only.

Written to `results/_analysis/figures_paper/tab_estimator_gap_pr_classic3.tex`.

In [3]:
from src.evaluation.benchmark_summary import cutoff_donor_points, donor_lopo_stats

lat = gaps                            # already restricted to TABLE_MODELS in cell 4

FMT = "{:.3f}"                        # one precision everywhere: levels, half-widths, gap
PM  = r"$\pm$"
NS3 = [50, 500, None]                 # None = the full panel, resolved PER DATASET
def ns_label(n): return "all" if n is None else str(n)

# BOLD: one single condition, on the ROUNDED value as it is displayed,
#   |round(B - A, 3)| >= BOLD_MIN_GAP
# not on the raw value. Otherwise two cells both showing "0.005" would be
# treated differently on the 4th decimal, which the reader does not see (that was the case of
# TuPro NB: +0.0052 bolded at n=50, -0.0046 not bolded at n=500, both displayed as 0.005).
# The bolding is thus checkable by eye on the table itself. A MAGNITUDE threshold, not a test.
BOLD_MIN_GAP = 0.005

# Multi-line dataset labels: the continuation is placed in the FOLLOWING ROWS of the
# block ("Tumor" on the n=50 row, "Profiler" on the n=500 row), not in the same cell.
# This is the only way to add no height: a p{} or a nested tabular would have
# made the n=50 row twice as tall, which digs a gap between n=50 and n=500. Here we
# use the rows that already exist, so the column stays aligned on "Kidney/Lung" and
# the line spacing does not move. A 3-row block therefore accepts up to 3 label lines.
# LaTeX only: the markdown render in cell 8 keeps DATASET_LABELS on a single line.
DATASET_TEX_LINES = {
    "Tumor Profiler":       ["Tumor", "Profiler"],
    "Kidney/Lung with TLS": ["Kidney/Lung", "with TLS"],
}

# The opening sentence is a human CLAIM, not a computation: recheck it if the data, the
# cohort or the threshold change. Everything else in the caption is generated from the same
# constants as the table, so it cannot drift out of sync with it.
CAPTION_LEAD = ("The two log1p estimators coincide for the Poisson head and separate only for "
                "the negative binomial.")
TABLE_LABEL  = "tab:estimator-gap"

# --- the TWO intervals, one per column type ----------------------------------------------------
# B and A carry the LEVEL interval: INTER-DONOR spread of the performance itself,
#     a property of the cohort (what this number would do in another patient). It is wide.
# B - A carries the PAIRED interval: each donor contributes B - A on ITS genes, so the
#     between-donor variation cancels and it is one to two orders tighter. That is the one
#     that says whether the gap is real, and putting it in the table is what stops the reader
#     from concluding, from the first two overlapping intervals, that it is not.
points = cutoff_donor_points(summaries, ns=NS3, metrics=("pearson", "pearson_logmean"),
                             rank_metric="pearson")
points = points[points["model"].isin(TABLE_MODELS)]

LEVEL = {}
for (ds, m, metric, n), g in points.groupby(["dataset", "model", "metric", "n"]):
    st = donor_lopo_stats(dict(zip(g["donor"], g["value"])))
    lo, hi = st["ci95_mean"]
    LEVEL[(ds, m, metric, int(n))] = (st["mean"], 0.5 * (hi - lo), st["K"])

print("donor Student-t half-widths on the LEVEL (columns B and A)")
for (ds, m, metric, n), (mean, half, K) in sorted(LEVEL.items()):
    print(f"   {ds:<12s} {m:<36s} {metric:<15s} n={n:<5d} "
          f"{mean:.4f} +/- {half:.4f}  (K={K})")
print()

def pick(sub, m, n):
    n = sub["n"].max() if n is None else n
    r = sub[(sub["model"] == m) & (sub["n"] == n)]
    return None if r.empty else r.iloc[0]

def level_parts(ds, m, metric, n):
    """(formatted level, formatted half-width)."""
    mean, half, _ = LEVEL.get((ds, m, metric, n), (float("nan"),) * 3)
    return FMT.format(mean), FMT.format(half)

audit = []     # printed twin of the table: one entry per row shown

def row_cells(ds, m, n):
    """[B, A, B - A]. B and A carry the level interval, B - A the PAIRED interval."""
    r = pick(lat[lat["dataset"] == ds], m, n)
    if r is None:
        return ["--", "--", "--"]
    nn = int(r["n"])
    b_mean, b_half = level_parts(ds, m, "pearson", nn)
    a_mean, a_half = level_parts(ds, m, "pearson_logmean", nn)
    b, a = b_mean + PM + b_half, a_mean + PM + a_half
    gap   = float(r["gap_mean"])
    shown = round(gap, 3)             # THE value that decides the bolding, the one the reader sees
    half  = 0.5 * (r["gap_ci_hi"] - r["gap_ci_lo"])
    # "-0.000" is meaningless: anything that rounds to zero is displayed without a sign.
    d = ("0.000" if shown == 0 else f"{shown:+.3f}") + PM + FMT.format(half)
    bold = abs(shown) >= BOLD_MIN_GAP
    audit.append((ds, m, nn, float(r["B_mean"]), float(r["A_mean"]),
                  gap, half, bool(r["excludes_zero_t"]), bold))
    if not bold:
        return [b, a, d]
    return [f"**{b}**", a, d] if shown > 0 else [b, f"**{a}**", d]

# --- the table: (dataset, head, cutoff) as rows ------------------------------------------------
header = ["Dataset", "Model", "$n$", "$B$", "$A$", "$B-A$"]
N_TEXT = 3
rows   = []
blocks = [(ds, m) for ds in DATASETS for m in TABLE_MODELS
          if not lat[lat["dataset"] == ds].empty]
for i, (ds, m) in enumerate(blocks):
    first_ds = (i == 0) or blocks[i - 1][0] != ds
    for j, n in enumerate(NS3):
        rows.append([DATASET_LABELS.get(ds, ds) if (first_ds and j == 0) else "",
                     MODEL_LABELS.get(m, m) if j == 0 else "",
                     ns_label(n)] + row_cells(ds, m, n))
    if i < len(blocks) - 1:
        rows.append("SEP")            # a bit of air between two heads; ignored in markdown

def to_latex(header, rows, n_text):
    def tex(c):    # the bolding is written in markdown by the builder; here it is translated
        return r"\textbf{" + c[2:-2] + "}" if c.startswith("**") and c.endswith("**") else c
    out = [r"\begin{tabular}{" + "l" * n_text + "r" * (len(header) - n_text) + "}", r"\toprule",
           " & ".join(header) + r" \\", r"\midrule"]
    pending = []                      # label lines to place in the following rows
    for r in rows:
        if r == "SEP":
            out.append(r"\addlinespace")
            pending = []              # a block never overflows onto the next one
            continue
        cells = list(r)
        if cells[0] in DATASET_TEX_LINES:
            lines = DATASET_TEX_LINES[cells[0]]
            cells[0], pending = lines[0], list(lines[1:])
        elif pending:
            cells[0], pending = pending[0], pending[1:]
        out.append(" & ".join(tex(c) for c in cells) + r" \\")
    return "\n".join(out + [r"\bottomrule", r"\end{tabular}"])

caption = (
    r"\textbf{" + CAPTION_LEAD + "} "
    r"$B$ is the predictive mean of the transform and $A$ the plug-in mean of the fitted mean, "
    r"both defined in the estimators appendix and both read on the "
    r"$\log(1+\text{counts per }10^{4})$ scale. Each entry averages the slide-aware Pearson "
    r"correlation over the first $n$ genes of the shared ordering and then over the held-out "
    r"donors. $B$ and $A$ carry the half-width of the donor Student-$t$ interval on that "
    r"accuracy, which is a between-donor quantity; $B-A$ carries the half-width of the paired "
    r"donor interval, each donor contributing $B-A$ over its own genes, which is why it is far "
    r"tighter and why two overlapping intervals in the first two columns are compatible with a "
    r"well determined difference. Bold marks the estimator ahead wherever the difference shown "
    r"reaches " f"${BOLD_MIN_GAP:g}$"
    r"; this is a magnitude cut on $B-A$ and not a significance test."
)

tex_full = "\n".join([
    r"\begin{table}[t]",
    r"\caption{" + caption + "}",
    r"\label{" + TABLE_LABEL + "}",
    r"\centering",
    to_latex(header, rows, N_TEXT),
    r"\end{table}",
])

# tex_full is NOT printed here: the last cell of the notebook holds only it, so that
# it is copyable without having to hunt for it in the middle of the diagnostics.
out = TEXDIR / f"tab_estimator_gap_{RUN_ID}.tex"
out.write_text(tex_full)
print(f"written: {out}\n")

# Printed twin of the table. The gap is given to 4 decimals, but BOLD follows the value
# ROUNDED to 3, the table's own: a +0.0046 being bolded is expected.
print("row by row (paired gap, its half-width, resolved, bolded)")
for ds, m, n, b, a, gap, half, res, bold in audit:
    print(f"   {ds:<12s} {m:<36s} n={n:<5d} B={b:.4f} A={a:.4f} "
          f"gap={gap:+.4f} (shown {round(gap, 3):+.3f}) +/- {half:.4f} "
          f"{'resolved' if res else '-':<7s} {'BOLD' if bold else ''}")
print(f"\n{sum(1 for *_, bold in audit if bold)} rows out of {len(audit)} bolded "
      f"(threshold {BOLD_MIN_GAP:g} on the displayed value)")

donor Student-t half-widths on the LEVEL (columns B and A)
   10x_TuPro    negativebinomial_hetero_noanchor_cp  pearson         n=50    0.5412 +/- 0.0694  (K=7)
   10x_TuPro    negativebinomial_hetero_noanchor_cp  pearson         n=500   0.3766 +/- 0.0697  (K=7)
   10x_TuPro    negativebinomial_hetero_noanchor_cp  pearson         n=3000  0.1655 +/- 0.0302  (K=7)
   10x_TuPro    negativebinomial_hetero_noanchor_cp  pearson_logmean n=50    0.5360 +/- 0.0726  (K=7)
   10x_TuPro    negativebinomial_hetero_noanchor_cp  pearson_logmean n=500   0.3812 +/- 0.0746  (K=7)
   10x_TuPro    negativebinomial_hetero_noanchor_cp  pearson_logmean n=3000  0.1609 +/- 0.0377  (K=7)
   10x_TuPro    poisson_cp                           pearson         n=50    0.5386 +/- 0.0682  (K=7)
   10x_TuPro    poisson_cp                           pearson         n=500   0.3863 +/- 0.0727  (K=7)
   10x_TuPro    poisson_cp                           pearson         n=3000  0.1653 +/- 0.0356  (K=7)
   10x_TuPro    poisson

## 3. The same table, rendered

The booktabs source above cannot be rendered as-is in a markdown cell: Jupyter's MathJax handles math, not `tabular`, `\toprule` or `\midrule`, which are LaTeX package commands. The cell below re-renders the SAME `(header, rows)` as a markdown table, `to_markdown` in place of `to_latex` and nothing else.

The bold is written in markdown syntax by the builder and translated to `\textbf` by `to_latex`, so it renders natively here and the two views cannot disagree about which cell is bold. Markdown has no two-level header, so a group label is folded into each of its sub-column labels (`50 B`, `50 A`, ...) where LaTeX stacks it in a `\multicolumn` row.

In [4]:
from IPython.display import Markdown, display

def to_markdown(header, rows, n_text):
    # The bolding is already written in markdown by the builder, nothing to translate here. The
    # "SEP" rows are a LaTeX typographic space (\addlinespace) with no markdown equivalent.
    align = [":---"] * n_text + ["---:"] * (len(header) - n_text)
    out = ["| " + " | ".join(header) + " |", "| " + " | ".join(align) + " |"]
    out += ["| " + " | ".join(r) + " |" for r in rows if r != "SEP"]
    return "\n".join(out)

display(Markdown(to_markdown(header, rows, N_TEXT)))

| Dataset | Model | $n$ | $B$ | $A$ | $B-A$ |
| :--- | :--- | :--- | ---: | ---: | ---: |
| Tumor Profiler | Poisson | 50 | 0.539$\pm$0.068 | 0.538$\pm$0.068 | 0.000$\pm$0.000 |
|  |  | 500 | 0.386$\pm$0.073 | 0.386$\pm$0.073 | 0.000$\pm$0.000 |
|  |  | all | 0.165$\pm$0.036 | 0.165$\pm$0.036 | 0.000$\pm$0.000 |
|  | Negative binomial | 50 | **0.541$\pm$0.069** | 0.536$\pm$0.073 | +0.005$\pm$0.007 |
|  |  | 500 | 0.377$\pm$0.070 | **0.381$\pm$0.075** | -0.005$\pm$0.011 |
|  |  | all | **0.166$\pm$0.030** | 0.161$\pm$0.038 | +0.005$\pm$0.011 |
| Kidney/Lung with TLS | Poisson | 50 | 0.257$\pm$0.129 | 0.256$\pm$0.129 | +0.001$\pm$0.000 |
|  |  | 500 | 0.136$\pm$0.085 | 0.135$\pm$0.084 | 0.000$\pm$0.000 |
|  |  | all | 0.039$\pm$0.027 | 0.039$\pm$0.027 | 0.000$\pm$0.000 |
|  | Negative binomial | 50 | 0.226$\pm$0.133 | 0.224$\pm$0.124 | +0.002$\pm$0.025 |
|  |  | 500 | **0.123$\pm$0.093** | 0.117$\pm$0.079 | +0.006$\pm$0.023 |
|  |  | all | **0.040$\pm$0.037** | 0.034$\pm$0.024 | +0.005$\pm$0.016 |
| NSCLC | Poisson | 50 | 0.221$\pm$0.041 | 0.221$\pm$0.041 | 0.000$\pm$0.000 |
|  |  | 500 | 0.140$\pm$0.026 | 0.140$\pm$0.026 | 0.000$\pm$0.000 |
|  |  | all | 0.053$\pm$0.007 | 0.053$\pm$0.007 | 0.000$\pm$0.000 |
|  | Negative binomial | 50 | **0.220$\pm$0.039** | 0.203$\pm$0.039 | +0.017$\pm$0.022 |
|  |  | 500 | **0.163$\pm$0.025** | 0.133$\pm$0.024 | +0.030$\pm$0.020 |
|  |  | all | **0.072$\pm$0.011** | 0.051$\pm$0.006 | +0.021$\pm$0.010 |

## 4. LaTeX, alone

Last cell, nothing but the complete float: `table` environment, caption, label, `tabular`. 

The caption is generated next to the table from the same constants, so the stated threshold can never drift from the bolding actually applied. Only `CAPTION_LEAD`, the opening claim in bold, is written by hand: re-read it if the cohort, the heads or the threshold change.

In [5]:
print(tex_full)

\begin{table}[t]
\caption{\textbf{The two log1p estimators coincide for the Poisson head and separate only for the negative binomial.} $B$ is the predictive mean of the transform and $A$ the plug-in mean of the fitted mean, both defined in the estimators appendix and both read on the $\log(1+\text{counts per }10^{4})$ scale. Each entry averages the slide-aware Pearson correlation over the first $n$ genes of the shared ordering and then over the held-out donors. $B$ and $A$ carry the half-width of the donor Student-$t$ interval on that accuracy, which is a between-donor quantity; $B-A$ carries the half-width of the paired donor interval, each donor contributing $B-A$ over its own genes, which is why it is far tighter and why two overlapping intervals in the first two columns are compatible with a well determined difference. Bold marks the estimator ahead wherever the difference shown reaches $0.005$; this is a magnitude cut on $B-A$ and not a significance test.}
\label{tab:estimator-gap